In [ ]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax.numpy as jnp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

In [ ]:
# flux_surface =  jax_sbgeom.flux_surfaces.FluxSurface.from_hdf5("/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4")
# s_mesh = jnp.linspace(0, 1, 10)
# tetra_mesh = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_tetrahedra(flux_surface, s_mesh, ToroidalExtent.half_module(flux_surface), 50,60)
# face_template = jsn.tools.mesh_tools._face_template(tetra_mesh[1].shape[1])
# geometric_domain = jsn.domain.GeometricDomain.from_mesh(tetra_mesh, face_template)
# quadrature_set   = jsn.quadrature_set.tn.create_tn_quadrature_set(4)
# pol_mesh = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface, jnp.linspace(0,1,10)**2, 0.52,  20)
# tri_face_template = jsn.tools.mesh_tools._face_template(pol_mesh[1].shape[1])
# pol_domain = jsn.domain.GeometricDomain.from_mesh(pol_mesh, tri_face_template)


In [ ]:
from basix import ElementFamily, CellType, LagrangeVariant
import basix # keep in mind entity dofs
from jax_sn.solution_domain.base_fem import compute_volume_matrices_linear_jacobian, Element
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex, _create_face_quadrature


element = BasixLagrangianSimplex(2,2)

In [ ]:
from finite_elements.test_finite_elements import _get_volume_matrix_from_old_generate_basis, _get_face_matrices_from_old_generate_basis

In [ ]:
_get_face_matrices_from_old_generate_basis("triangle", 2)

In [ ]:
def compute_Cij(element : Element) -> onp.ndarray:
    face_quadratures = element.face_quadratures
    
    for face in range(element.n_faces):
        p_i, w_i = face_quadratures[face]
        h_face = element.tabulate_basis_functions_first_derivative(p_i)[0,:,:] #[n_points, n_basis]
        print(f"Face {face}:")
        
        hihj = onp.einsum("qi,qj->qij", h_face, h_face)  #[n_quadrature_points, n_basis, n_basis]
        mass_matrix = onp.einsum("qij,q->ij", hihj, w_i)  #[n_basis, n_basis]
        print(mass_matrix)
        
        

    
    


compute_Cij(element)

In [ ]:
basix.cell.geometry(CellType.triangle)

In [ ]:
basix.cell.geometry(CellType.hexahedron)

In [ ]:
from mpi4py import MPI
from dolfinx.mesh import create_box, CellType
from dolfinx.fem import FunctionSpace, functionspace
import ufl

mesh = create_box(MPI.COMM_WORLD,
                  [[0, 0, 0], [1, 1, 1]],
                  [1, 1, 1],
                  CellType.hexahedron)
msh_cell_type = mesh.basix_cell()

Ve = basix.ufl.element("Lagrange", msh_cell_type, 1)
V = functionspace(mesh, Ve)
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
a = ufl.inner(u, v)*ufl.ds
from dolfinx.fem import form
f = form(a)
print("Form compiled, function space created successfully.")
# -----------------------------
#cpp_code = f._cpp_code  # This triggers FFCx code generation
import ffcx 



aa = ffcx.compiler.compile_ufl_objects([a], {"scalar_type" : "double", "sum_factorization":False, "table_rtol" : 0.2, "table_atol" : 1e-4})
print(aa[1])
with open("hexahedron_facet_integral.cpp", "w") as f:
    f.write(aa[0])
    f.write(aa[1])
# with open("hexahedron_facet_integral.cpp", "w") as f:
#     f.write(cpp_code)

print("C++ kernel generated! Look for X[0], X[1], X[2] lines for facet mapping.")

In [ ]:
from dolfinx import mesh, fem
import ufl
import numpy as np
from mpi4py import MPI

# -----------------------------
# 1. Create a simple hexahedral mesh
# -----------------------------
msh = mesh.create_box(MPI.COMM_WORLD,
                      [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]],
                      [1, 1, 1],  # 1 cell per dimension
                      mesh.CellType.hexahedron)

# -----------------------------
# 2. Define a function space
# -----------------------------
V = fem.FunctionSpace(msh, ("CG", 1))  # Continuous Lagrange, degree 1

# -----------------------------
# 3. Define a facet integral
# -----------------------------
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Integrate u*v over exterior facets
a = ufl.inner(u, v) * ufl.ds

# -----------------------------
# 4. Create the compiled form
# -----------------------------
form = fem.form(a)

# -----------------------------
# 5. Dump the generated C++ kernel
# -----------------------------
cpp_code = form._cpp_code  # This triggers FFCx code generation

# Optionally save to a file
with open("hexahedron_facet_integral.cpp", "w") as f:
    f.write(cpp_code)

print("C++ kernel generated! Look for X[0], X[1], X[2] lines for facet mapping.")


In [ ]:
basix.cell.sub_entity_connectivity(CellType.triangle)

In [ ]:
basix._basixcpp.subentity_types(CellType.triangle)

In [ ]:
basix._basixcpp.sub_entity_geometry(CellType.triangle, 1,2)

In [ ]:
basix.topology(CellType.triangle)

In [ ]:
basix.cell.sub_entity_connectivity(CellType.triangle)[1][0]

In [ ]:
print(basix.cell.subentity_types(CellType.interval)[-2])
print(basix.cell.subentity_types(CellType.triangle)[-2])
print(basix.cell.subentity_types(CellType.tetrahedron)[-2])

basix.cell.sub_entity_connectivity(CellType.triangle)

In [ ]:
element.non_discontinuous_element.entity_dofs